# Validation Test Evaluation - Rising Bubble Benchmark

This notebook and the corresponding simulation setup notebook (RisingBubble2D_Run.ipynb) are part of published results (cf. 7.3) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853). We compare our results against the rising bubble benchmark test case established by Hysing et al ( https://doi.org/10.1002/fld.1934)

In [1]:
//#r "BoSSSpad.dll"
#r "..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
using System.IO;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [3]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [4]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [5]:
var myBatch = ExecutionQueues[1];
if(userName.Equals(@"FDY\JenkinsCI"))
    myBatch = ExecutionQueues[4];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [6]:
BoSSSshell.WorkflowMgm.Init("RisingBubble2D", myBatch);

Project name is set to 'RisingBubble2D'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\RisingBubble2D'.


In [7]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_StokesExt_ReInit300_semiImplicit_dt0.001_testcase1	7/9/2025 3:58:03 PM	cf198209...
#1: RisingBubble2D	RB2D_fullDomain_10x20AMR0_k3_StokesExt_ReInit100_semiImplicit_dt0.001_testcase1	7/9/2025 3:55:54 PM	b974a59a...
#2: RisingBubble2D	RB2D_fullDomain_10x20AMR0_k3_StokesExt_ReInit100_semiImplicit_dt0.0025_testcase1	7/9/2025 3:14:43 PM	c749094c...
#3: RisingBubble2D	RB2D_fullDomain_10x20AMR0_k3_StokesExt_ReInit50_semiImplicit_testcase1	6/30/2025 12:07:18 PM	4fbe849b...
#4: RisingBubble2D	RB2D_fullDomain_10x20AMR0_k3_Newton_StokesExt_ReInit50_Stokes_testcase1	6/30/2025 9:53:14 AM	ed3df836...
#5: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_Newton_StokesExt_ReInit100_SemiImplicit_testcase1*	6/30/2025 9:48:42 AM	10e76ce9...
#6: RisingBubble2D	RB2D_fullDomain_10x20AMR0_k3_Newton_StokesExt_ReInit100_SemiImplicit_halfdt_testcase1	6/30/2025 9:00:21 AM	c2417c2f...
#7: RisingBubble2D	RB2D_fullDomain_10x20AMR0_k3_Newton_StokesExt_ReInit50_SemiImplicit_t

In [8]:
string[] studyNames = {"RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1",
                       "RB2D_fullDomain_20x40AMR0_k3_StokesExt_ReInit300_semiImplicit_dt0.001_testcase1"};

In [9]:
static string[] LegendNames = {"paper", "semiImplicit"};

In [10]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [11]:
foreach (string studyName in studyNames) {
    Console.WriteLine("Searching for: " + studyName);
    foreach (var s in sessions) {
        if (s.Name.Contains(studyName)) {
            evalSess.Add(s);
            Console.WriteLine("found");
            break;
        }
    }
}
evalSess

Searching for: RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1
found
Searching for: RB2D_fullDomain_20x40AMR0_k3_StokesExt_ReInit300_semiImplicit_dt0.001_testcase1
found


#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1	8/30/2024 9:35:54 AM	46f02668...
#1: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_StokesExt_ReInit300_semiImplicit_dt0.001_testcase1	7/9/2025 3:58:03 PM	cf198209...


In [12]:
NUnit.Framework.Assert.IsTrue(evalSess.Count == studyNames.Length, "Not all requested sessions were found with the specified study name.");

# Evaluation - rising bubble benchmark quantities

In order to compare the different numerical methods three scalar measures for the temporal evolution of the rising bubble are introduced. 
First, the center of mass is considered given as
>$$
\vec{x}_c = \frac{ \int_{\mathfrak{A}} \vec{x} dV }{ \int_{\mathfrak{A}} 1 dV}.
$$<
The second measure is denoted as circularity and defined by
>$$
\cancel{c} = \frac{\text{perimeter of area-equivalent circle}}{\text{perimeter of bubble}}.
$$<
The circularity is $\cancel{c} = 1$ for a perfectly circular bubble and gets smaller for more deformed bubble shapes. 
The third measure is the mean bubble velocity
>$$
\vec{u}_c = \frac{ \int_{\mathfrak{A}} \vec{u} dV }{ \int_{\mathfrak{A}} 1 dV },
$$< 
where the component in $y$-direction is denoted as the rise velocity $V_c$. The error quantification is done via the $l_2$-error norm (44) and the corresponding ROC values (45). 
Note that in the original benchmark paper furthermore $l_1$  and $l_\infty$-error norms are considered. 
Besides the three scalar quantities the terminal shape of the bubble at t = 3 is compared

In [13]:
public static void PerformPostProcessing(ISessionInfo evalS) {

    // set up log 
    var db = databases.Pick(0);
    TextWriter Log = db.Controller.DBDriver.FsDriver.GetNewLog(RisingBubble2DBenchmarkQuantities.LogfileName, evalS.ID);
    string header = String.Format("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}", "#timestep", "time", "area", "center of mass - x", "center of mass - y", "circularity", "mean velocity - x", "mean velocity - y");
    Log.WriteLine(header);
    Log.Flush();

    // perform postprocessing for each time step
    foreach(var tStep in evalS.Timesteps) {

        // set up LsTrk and velocity fields   
        SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
        GridData grdData = (GridData)phi.GridDat; 
        LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
        levelSet.Acc(1.0, phi); 
        LevelSetTracker LsTrk = new LevelSetTracker(grdData, CutCellQuadratureMethod.Saye,  1, new string[] {"A", "B"}, levelSet);
        LsTrk.UpdateTracker(tStep.PhysicalTime);

        int D = grdData.SpatialDimension;
        XDGField[] VelocityFields = new XDGField[D];
        VelocityFields[0] = (XDGField)tStep.GetField("VelocityX");
        VelocityFields[1] = (XDGField)tStep.GetField("VelocityY");

        // compute benchmark quantities
        var R = RisingBubbleBenchmarkQuantitites.ComputeBenchmarkQuantities_RisingBubble2D(LsTrk, VelocityFields);

        // write line to log
        string line = String.Format("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}", tStep.TimeStepNumber.MajorNumber, tStep.PhysicalTime, 
                        R.area, R.centerX, R.centerY, R.circularity, R.MeanVelocityX, R.MeanVelocityY);
        Log.WriteLine(line);
        Log.Flush();
    }

    // Dispose
    Log.Close();
    Log.Dispose();

}

In [14]:
List<Plot2Ddata> evalData = evalSess.ReadLogDataForXNSE(RisingBubble2DBenchmarkQuantities.LogfileName);
// PerformPostProcessing(eval)

Element at 0: time vs area
Element at 1: time vs center of mass - x
Element at 2: time vs center of mass - y
Element at 3: time vs circularity
Element at 4: time vs mean velocity - x
Element at 5: time vs mean velocity - y


In [15]:
List<Plot2Ddata> benchmarkData = new List<Plot2Ddata>();
// 1: area 2: circularity 3: center y 4: rise velocity
benchmarkData.Add(evalData.ElementAt(0));
benchmarkData.Add(evalData.ElementAt(3));
benchmarkData.Add(evalData.ElementAt(2));
benchmarkData.Add(evalData.ElementAt(5));

In [16]:
public static Plot2Ddata GetBenchmarkQuantityOverTime_Plot2Ddata(List<Plot2Ddata> data, int index, string name = "") {
    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = data.ElementAt(index).Xlabel;
    plt.Ylabel = (!name.IsEmptyOrWhite()) ? name : data.ElementAt(index).Ylabel;
    
    var datGroups = data.ElementAt(index).dataGroups;
    for (int i = 0; i < datGroups.Count(); i++) {

        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(i+1); //LineColors.Blue;

        plt.AddDataGroup(LegendNames[i], datGroups[i].Abscissas, datGroups[i].Values, fmt);
    }

    plt.ShowLegend = true;
    
    return plt;
}

In [17]:
GetBenchmarkQuantityOverTime_Plot2Ddata(evalData, 2).ToGnuplot().PlotSVG()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 1.1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 center of mass - y 
 
 
 
 
 time 
 
 
 
 
 paper 
 
 
 
 
 paper 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M544.1,57.1 L597.5,57.1 M71.9,470.1 L72.4,470.1 L72.8,470.1 L73.3,470.1 L73.7,470.1 L74.2,470.1
 L74.6,470.0 L75.1,470.0 L75.5,470.0 L76.0,470.0 L76.4,470.0 L76.9,470.0 L77.3,470.0 L77.8,470.0
 L78.2,469.9 L78.7,469.9 L79.1,469.9 L79.6,469.9 L80.0,469.8 L80.5,469.8 L80.9,469.8 L81.4,469.8
 L81.8,469.7 L82.3,469.7 L82.8,469.7 L83.2,469.6 L83.7,469.6 L84.1,469.5 L84.6,469.5 L85.0,469.5
 L85.5,469.4 L85.9,469.4 L86.4,469.3 L86.8,469.3 L87.3,469.2 L87.7,469.2 L88.2,469.1 L88.6,469.1
 L89.1,469.0 L89.5,469.0 L90.0,468.9 L90.4,468.9 L90.9,468.8 L91.3,468.7 L91.8,468.7 L92.2,468.6
 L92.7,468.5 L93.2,468.5 L93.6,468.4 L94.1,468.3 L94.5,468.3 L95.0,468.2 L95.4,468.1 L95.9,468.0
 L96.3,468.0 L96.8,467.9 L97.2,467.8 L97.7,467.7 L98.1,467.7 L98.6,467.6 L99.0,467.5 L99.5,467.4
 L99.9,467.3 L100.4,467.2 L100.8,467.1 L101.3,467.0 L101.7,467.0 L102.2,466.9 L102.6,466.8 L103.1,466.7
 L103.5,466.6 L104.0,466.5 L104.5,466.4 L104.9,466.3 L105.4,466.2 L105.8,466.1 L106.3,466.0 L106.7,465.9
 L107.2,465.7 L107.6,465.6 L108.1,465.5 L108.5,465.4 L109.0,465.3 L109.4,465.2 L109.9,465.1 L110.3,465.0
 L110.8,464.8 L111.2,464.7 L111.7,464.6 L112.1,464.5 L112.6,464.4 L113.0,464.2 L113.5,464.1 L113.9,464.0
 L114.4,463.9 L114.9,463.7 L115.3,463.6 L115.8,463.5 L116.2,463.3 L116.7,463.2 L117.1,463.1 L117.6,462.9
 L118.0,462.8 L118.5,462.6 L118.9,462.5 L119.4,462.4 L119.8,462.2 L120.3,462.1 L120.7,461.9 L121.2,461.8
 L121.6,461.6 L122.1,461.5 L122.5,461.3 L123.0,461.2 L123.4,461.0 L123.9,460.9 L124.3,460.7 L124.8,460.6
 L125.3,460.4 L125.7,460.3 L126.2,460.1 L126.6,459.9 L127.1,459.8 L127.5,459.6 L128.0,459.5 L128.4,459.3
 L128.9,459.1 L129.3,459.0 L129.8,458.8 L130.2,458.6 L130.7,458.5 L131.1,458.3 L131.6,458.1 L132.0,457.9
 L132.5,457.8 L132.9,457.6 L133.4,457.4 L133.8,457.2 L134.3,457.1 L134.7,456.9 L135.2,456.7 L135.7,456.5
 L136.1,456.3 L136.6,456.1 L137.0,456.0 L137.5,455.8 L137.9,455.6 L138.4,455.4 L138.8,455.2 L139.3,455.0
 L139.7,454.8 L140.2,454.6 L140.6,454.4 L141.1,454.2 L141.5,454.0 L142.0,453.8 L142.4,453.6 L142.9,453.4
 L143.3,453.2 L143.8,453.0 L144.2,452.8 L144.7,452.6 L145.1,452.4 L145.6,452.2 L146.0,452.0 L146.5,451.8
 L147.0,451.6 L147.4,451.4 L147.9,451.2 L148.3,451.0 L148.8,450.7 L149.2,450.5 L149.7,450.3 L150.1,450.1
 L150.6,449.9 L151.0,449.7 L151.5,449.4 L151.9,449.2 L152.4,449.0 L152.8,448.8 L153.3,448.6 L153.7,448.3
 L154.2,448.1 L154.6,447.9 L155.1,447.7 L155.5,447.4 L156.0,447.2 L156.4,447.0 L156.9,446.7 L157.4,446.5
 L157.8,446.3 L158.3,446.0 L158.7,445.8 L159.2,445.6 L159.6,445.3 L160.1,445.1 L160.5,444.9 L161.0,444.6
 L161.4,444.4 L161.9,444.1 L162.3,443.9 L162.8,443.7 L163.2,443.4 L163.7,443.2 L164.1,442.9 L164.6,442.7
 L165.0,442.4 L165.5,442.2 L165.9,441.9 L166.4,441.7 L166.8,441.4 L167.3,441.2 L167.8,440.9 L168.2,440.7
 L168.7,440.4 L169.1,440.2 L169.6,439.9 L170.0,439.6 L170.5,439.4 L170.9,439.1 L171.4,438.9 L171.8,438.6
 L172.3,438.3 L172.7,438.1 L173.2,437.8 L173.6,437.6 L174.1,437.3 L174.5,437.0 L175.0,436.8 L175.4,436.5
 L175.9,436.2 L176.3,436.0 L176.8,435.7 L177.2,435.4 L177.7,435.2 L178.2,434.9 L178.6,434.6 L179.1,434.3
 L179.5,434.1 L180.0,433.8 L180.4,433.5 L180.9,433.2 L181.3,433.0 L181.8,432.7 L182.2,432.4 L182.7,432.1
 L183.1,431.8 L183.6,431.6 L184.0,431.3 L184.5,431.0 L184.9,430.7 L185.

In [18]:
GetBenchmarkQuantityOverTime_Plot2Ddata(evalData, 3).ToGnuplot().PlotSVG()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 0.91 
 
 
 
 
 0.92 
 
 
 
 
 0.93 
 
 
 
 
 0.94 
 
 
 
 
 0.95 
 
 
 
 
 0.96 
 
 
 
 
 0.97 
 
 
 
 
 0.98 
 
 
 
 
 0.99 
 
 
 
 
 1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 circularity 
 
 
 
 
 time 
 
 
 
 
 paper 
 
 
 
 
 paper 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M544.1,57.1 L597.5,57.1 M80.2,36.1 L80.6,36.1 L81.1,36.1 L81.5,36.1 L82.0,36.1 L82.4,36.1
 L82.9,36.1 L83.3,36.1 L83.8,36.1 L84.2,36.1 L84.7,36.1 L85.1,36.1 L85.6,36.1 L86.0,36.1
 L86.5,36.1 L86.9,36.1 L87.3,36.1 L87.8,36.1 L88.2,36.1 L88.7,36.1 L89.1,36.1 L89.6,36.1
 L90.0,36.1 L90.5,36.1 L90.9,36.1 L91.4,36.1 L91.8,36.1 L92.3,36.1 L92.7,36.1 L93.2,36.1
 L93.6,36.1 L94.0,36.1 L94.5,36.1 L94.9,36.1 L95.4,36.1 L95.8,36.1 L96.3,36.1 L96.7,36.1
 L97.2,36.1 L97.6,36.1 L98.1,36.1 L98.5,36.1 L99.0,36.1 L99.4,36.1 L99.9,36.1 L100.3,36.1
 L100.7,36.1 L101.2,36.1 L101.6,36.1 L102.1,36.1 L102.5,36.1 L103.0,36.1 L103.4,36.1 L103.9,36.1
 L104.3,36.1 L104.8,36.1 L105.2,36.1 L105.7,36.1 L106.1,36.1 L106.5,36.1 L107.0,36.1 L107.4,36.1
 L107.9,36.1 L108.3,36.1 L108.8,36.1 L109.2,36.1 L109.7,36.1 L110.1,36.1 L110.6,36.1 L111.0,36.1
 L111.5,36.1 L111.9,36.1 L112.4,36.1 L112.8,36.1 L113.2,36.1 L113.7,36.1 L114.1,36.1 L114.6,36.1
 L115.0,36.1 L115.5,36.1 L115.9,36.1 L116.4,36.1 L116.8,36.1 L117.3,36.1 L117.7,36.1 L118.2,36.1
 L118.6,36.1 L119.1,36.1 L119.5,36.1 L119.9,36.1 L120.4,36.1 L120.8,36.1 L121.3,36.1 L121.7,36.1
 L122.2,36.1 L122.6,36.1 L123.1,36.1 L123.5,36.1 L124.0,36.1 L124.4,36.1 L124.9,36.1 L125.3,36.1
 L125.8,36.1 L126.2,36.1 L126.6,36.1 L127.1,36.1 L127.5,36.1 L128.0,36.1 L128.4,36.1 L128.9,36.1
 L129.3,36.1 L129.8,36.1 L130.2,36.1 L130.7,36.1 L131.1,36.1 L131.6,36.1 L132.0,36.1 L132.5,36.1
 L132.9,36.1 L133.3,36.1 L133.8,36.1 L134.2,36.1 L134.7,36.2 L135.1,36.2 L135.6,36.2 L136.0,36.2
 L136.5,36.2 L136.9,36.2 L137.4,36.2 L137.8,36.2 L138.3,36.2 L138.7,36.2 L139.2,36.2 L139.6,36.2
 L140.0,36.2 L140.5,36.2 L140.9,36.2 L141.4,36.2 L141.8,36.2 L142.3,36.2 L142.7,36.2 L143.2,36.2
 L143.6,36.2 L144.1,36.2 L144.5,36.2 L145.0,36.2 L145.4,36.2 L145.9,36.2 L146.3,36.2 L146.7,36.2
 L147.2,36.2 L147.6,36.2 L148.1,36.2 L148.5,36.3 L149.0,36.3 L149.4,36.3 L149.9,36.3 L150.3,36.3
 L150.8,36.3 L151.2,36.3 L151.7,36.3 L152.1,36.3 L152.5,36.3 L153.0,36.3 L153.4,36.3 L153.9,36.3
 L154.3,36.3 L154.8,36.3 L155.2,36.4 L155.7,36.4 L156.1,36.4 L156.6,36.4 L157.0,36.4 L157.5,36.4
 L157.9,36.4 L158.4,36.4 L158.8,36.4 L159.2,36.4 L159.7,36.5 L160.1,36.5 L160.6,36.5 L161.0,36.5
 L161.5,36.5 L161.9,36.5 L162.4,36.5 L162.8,36.6 L163.3,36.6 L163.7,36.6 L164.2,36.6 L164.6,36.6
 L165.1,36.6 L165.5,36.7 L165.9,36.7 L166.4,36.7 L166.8,36.7 L167.3,36.7 L167.7,36.8 L168.2,36.8
 L168.6,36.8 L169.1,36.8 L169.5,36.8 L170.0,36.9 L170.4,36.9 L170.9,36.9 L171.3,36.9 L171.8,37.0
 L172.2,37.0 L172.6,37.0 L173.1,37.1 L173.5,37.1 L174.0,37.1 L174.4,37.2 L174.9,37.2 L175.3,37.2
 L175.8,37.3 L176.2,37.3 L176.7,37.3 L177.1,37.4 L177.6,37.4 L178.0,37.4 L178.5,37.5 L178.9,37.5
 L179.3,37.6 L179.8,37.6 L180.2,37.7 L180.7,37.7 L181.1,37.8 L181.6,37.8 L182.0,37.9 L182.5,37.9
 L182.9,38.0 L183.4,38.0 L183.8,38.1 L184.3,38.1 L184.7,38.2 L185.2,38.2 L185.6,38.3 L186.0,38.4
 L186.5,38.4 L186.9,38.5 L187.4,38.6 L187.8,38.6 L188.3,38.7 L188.7,38.8 L189.2,38.9 L189.6,38.9
 L190.1,39.0 L190.5,39.1 L191.0,39.2 L191.4,39.3 L191.9,39.3 L192.3,39.4 L192.7,39.5 L193.2,39.6
 L193.6,39.7 L194.1,39.8 L194.5,39.9 L195.0,40.0 L195.4,40.1 L195.9,40.2 L196.3,40.3 L196.8,40.4
 L197.2,40.5 L197.7,40.6 L198.1

In [19]:
GetBenchmarkQuantityOverTime_Plot2Ddata(evalData, 5, "rise velocity").ToGnuplot().PlotSVG()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rise velocity 
 
 
 
 
 time 
 
 
 
 
 paper 
 
 
 
 
 paper 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M544.1,57.1 L597.5,57.1 M80.2,542.4 L80.6,540.2 L81.1,538.1 L81.5,535.9 L82.0,533.8 L82.4,531.6
 L82.9,529.5 L83.3,527.4 L83.8,525.3 L84.2,523.2 L84.7,521.1 L85.1,519.0 L85.6,516.9 L86.0,514.8
 L86.5,512.8 L86.9,510.7 L87.3,508.6 L87.8,506.6 L88.2,504.5 L88.7,502.5 L89.1,500.4 L89.6,498.4
 L90.0,496.4 L90.5,494.3 L90.9,492.3 L91.4,490.3 L91.8,488.3 L92.3,486.3 L92.7,484.3 L93.2,482.3
 L93.6,480.3 L94.0,478.3 L94.5,476.4 L94.9,474.4 L95.4,472.4 L95.8,470.4 L96.3,468.5 L96.7,466.5
 L97.2,464.6 L97.6,462.6 L98.1,460.7 L98.5,458.8 L99.0,456.8 L99.4,454.9 L99.9,453.0 L100.3,451.0
 L100.7,449.1 L101.2,447.2 L101.6,445.3 L102.1,443.4 L102.5,441.5 L103.0,439.6 L103.4,437.7 L103.9,435.8
 L104.3,434.0 L104.8,432.1 L105.2,430.2 L105.7,428.3 L106.1,426.5 L106.5,424.6 L107.0,422.8 L107.4,420.9
 L107.9,419.0 L108.3,417.2 L108.8,415.4 L109.2,413.5 L109.7,411.7 L110.1,409.9 L110.6,408.0 L111.0,406.2
 L111.5,404.4 L111.9,402.6 L112.4,400.8 L112.8,399.0 L113.2,397.2 L113.7,395.4 L114.1,393.6 L114.6,391.8
 L115.0,390.0 L115.5,388.2 L115.9,386.4 L116.4,384.7 L116.8,382.9 L117.3,381.1 L117.7,379.4 L118.2,377.6
 L118.6,375.8 L119.1,374.1 L119.5,372.3 L119.9,370.6 L120.4,368.9 L120.8,367.1 L121.3,365.4 L121.7,363.7
 L122.2,361.9 L122.6,360.2 L123.1,358.5 L123.5,356.8 L124.0,355.1 L124.4,353.4 L124.9,351.7 L125.3,350.0
 L125.8,348.3 L126.2,346.6 L126.6,344.9 L127.1,343.2 L127.5,341.5 L128.0,339.9 L128.4,338.2 L128.9,336.5
 L129.3,334.9 L129.8,333.2 L130.2,331.5 L130.7,329.9 L131.1,328.2 L131.6,326.6 L132.0,325.0 L132.5,323.3
 L132.9,321.7 L133.3,320.1 L133.8,318.4 L134.2,316.8 L134.7,315.2 L135.1,313.6 L135.6,312.0 L136.0,310.4
 L136.5,308.8 L136.9,307.2 L137.4,305.6 L137.8,304.0 L138.3,302.4 L138.7,300.8 L139.2,299.2 L139.6,297.7
 L140.0,296.1 L140.5,294.5 L140.9,293.0 L141.4,291.4 L141.8,289.9 L142.3,288.3 L142.7,286.8 L143.2,285.2
 L143.6,283.7 L144.1,282.2 L144.5,280.6 L145.0,279.1 L145.4,277.6 L145.9,276.1 L146.3,274.6 L146.7,273.0
 L147.2,271.5 L147.6,270.0 L148.1,268.6 L148.5,267.1 L149.0,265.6 L149.4,264.1 L149.9,262.6 L150.3,261.1
 L150.8,259.7 L151.2,258.2 L151.7,256.8 L152.1,255.3 L152.5,253.8 L153.0,252.4 L153.4,251.0 L153.9,249.5
 L154.3,248.1 L154.8,246.7 L155.2,245.2 L155.7,243.8 L156.1,242.4 L156.6,241.0 L157.0,239.6 L157.5,238.2
 L157.9,236.8 L158.4,235.4 L158.8,234.0 L159.2,232.6 L159.7,231.2 L160.1,229.9 L160.6,228.5 L161.0,227.1
 L161.5,225.8 L161.9,224.4 L162.4,223.1 L162.8,221.7 L163.3,220.4 L163.7,219.1 L164.2,217.7 L164.6,216.4
 L165.1,215.1 L165.5,213.8 L165.9,212.5 L166.4,211.1 L166.8,209.8 L167.3,208.5 L167.7,207.3 L168.2,206.0
 L168.6,204.7 L169.1,203.4 L169.5,202.1 L170.0,200.9 L170.4,199.6 L170.9,198.4 L171.3,197.1 L171.8,195.9
 L172.2,194.6 L172.6,193.4 L173.1,192.2 L173.5,190.9 L174.0,189.7 L174.4,188.5 L174.9,187.3 L175.3,186.1
 L175.8,184.9 L176.2,183.7 L176.7,182.5 L177.1,181.3 L177.6,180.2 L178.0,179.0 L178.5,177.8 L178.9,176.7
 L179.3,175.5 L179.8,174.4 L180.2,173.2 L180.7,172.1 L181.1,171.0 L181.6,169.8 L182.0,168.7 L182.5,167.6
 L182.9,166.5 L183.4,165.4 L183.8,164.3 L184.3,163.2 L184.7,162.1 L185.2,161.0 L185.6,159.9 L186.0,158.9
 L186.5,157.8 L186.9,156.7 L187.4,155.7 L187.8,154.6 L188.3,153.6 L188.7,152.6 L189.2,151.5 L189.6,150.5
 L190.1,149.5 L190.5,148.5 L191.0,147.5 L191.4,146.5 L191.9,145.5 L192.3,144.5 L192.7,143.5 L193.2,142.

## Compare to Reference data (FEATFLOW benchmark)

In Hysing et al. an extensive dataset is provided by the following three research groups: 
TP2D (transport phenomena in 2D), FreeLIFE (free-surface library of finite element), and MoonMD (mathematics and object-oriented numerics in Magdeburg). 
For details on the methodology of each solver the reader is referred to the benchmark paper. 
The plotted data of the benchmark groups are taken from http://www.featflow.de/en/benchmarks/cfdbenchmarking/bubble/bubble_reference.html. 
Furthermore, we compare in the following some results with the work of Heimann et al., where the unfitted DG method with a piecewise linear approximation of the interface is employed.

In [20]:
string caseName = "testcase1";

In [21]:
// g1 TU Dortmund (TP2D); g2 EPFL Lausanne (FreeLIFE); g3 Uni Magdebug (MooNMD)
string[] groups = new string[] {"TP2D", "FreeLIFE", "MooNMD"};
int[] datLvl;
string datCase;
if(caseName.Equals("testcase1")) {
    datCase = "c1g";
    datLvl  = new int[] {7, 3, 4};    // testcase 1
} 
if(caseName.Equals("testcase2")) {     
    datCase = "c2g";
    datLvl  = new int[] {8, 3, 4};    // testcase 2
}
Plot2Ddata[,] dataRef = new Plot2Ddata[4,3];
for (int grp = 1; grp <= groups.Length; grp++) {
    Plot2Ddata[] datSet = new Plot2Ddata[4];
    // 1: area 2: circularity 3: center y 4: rise velocity
    for (int j = 0; j < 4; j++) {
        datSet[j] = new Plot2Ddata();
    }

    string dat  = datCase+grp+"l"+datLvl[grp - 1]+".txt";
    string path = (userName.Equals(@"FDY\JenkinsCI")) ? dat : @"Featflow_referenceData\" + dat;
        string[] lines = File.ReadAllLines(path);
        double[] time = new double[lines.Length];
        double[][] valueDat = new double[4][];
        for(int j = 0; j < 4; j++)
            valueDat[j] = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            //var datString = lines[i].Split(new string[] {" "}, StringSplitOptions.RemoveEmptyEntries);
            //Console.WriteLine("num split strings at 0: {0}", datString[0]);
            time[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            for (int j = 0; j < 4; j++) {
                valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
            }
        }        
        // Build DataSet
        for (int j = 0; j < 4; j++) {
            string datName = groups[grp-1]+"-l"+datLvl[grp - 1];
            datSet[j] = (new Plot2Ddata(new KeyValuePair<string, double[][]>(datName, new double[][] { time, valueDat[j] })));
        }
    
    for (int j = 0; j < 4; j++) {
        dataRef[j,grp-1] = datSet[j];
    }
}

In [22]:
public static Plot2Ddata GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(List<Plot2Ddata> data, int index, Plot2Ddata[,] refData,  string name = "") {

    Plot2Ddata plt = GetBenchmarkQuantityOverTime_Plot2Ddata(data, index, name);

    for (int i = 0; i < refData.GetLength(1); i++) {
        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.DotDashed;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(9-i);
        
        // Add reference data
        plt.AddDataGroup(refData[index, i].dataGroups[0], fmt);
    }
    
    return plt;
}

In [23]:
GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(benchmarkData, 1, dataRef).ToGnuplot().PlotSVG()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 0.91 
 
 
 
 
 0.92 
 
 
 
 
 0.93 
 
 
 
 
 0.94 
 
 
 
 
 0.95 
 
 
 
 
 0.96 
 
 
 
 
 0.97 
 
 
 
 
 0.98 
 
 
 
 
 0.99 
 
 
 
 
 1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 circularity 
 
 
 
 
 time 
 
 
 
 
 paper 
 
 
 
 
 paper 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M544.1,57.1 L597.5,57.1 M80.2,36.1 L80.6,36.1 L81.0,36.1 L81.3,36.1 L81.7,36.1 L82.1,36.1
 L82.5,36.1 L82.9,36.1 L83.3,36.1 L83.6,36.1 L84.0,36.1 L84.4,36.1 L84.8,36.1 L85.2,36.1
 L85.6,36.1 L85.9,36.1 L86.3,36.1 L86.7,36.1 L87.1,36.1 L87.5,36.1 L87.9,36.1 L88.2,36.1
 L88.6,36.1 L89.0,36.1 L89.4,36.1 L89.8,36.1 L90.2,36.1 L90.5,36.1 L90.9,36.1 L91.3,36.1
 L91.7,36.1 L92.1,36.1 L92.4,36.1 L92.8,36.1 L93.2,36.1 L93.6,36.1 L94.0,36.1 L94.4,36.1
 L94.7,36.1 L95.1,36.1 L95.5,36.1 L95.9,36.1 L96.3,36.1 L96.7,36.1 L97.0,36.1 L97.4,36.1
 L97.8,36.1 L98.2,36.1 L98.6,36.1 L99.0,36.1 L99.3,36.1 L99.7,36.1 L100.1,36.1 L100.5,36.1
 L100.9,36.1 L101.3,36.1 L101.6,36.1 L102.0,36.1 L102.4,36.1 L102.8,36.1 L103.2,36.1 L103.6,36.1
 L103.9,36.1 L104.3,36.1 L104.7,36.1 L105.1,36.1 L105.5,36.1 L105.8,36.1 L106.2,36.1 L106.6,36.1
 L107.0,36.1 L107.4,36.1 L107.8,36.1 L108.1,36.1 L108.5,36.1 L108.9,36.1 L109.3,36.1 L109.7,36.1
 L110.1,36.1 L110.4,36.1 L110.8,36.1 L111.2,36.1 L111.6,36.1 L112.0,36.1 L112.4,36.1 L112.7,36.1
 L113.1,36.1 L113.5,36.1 L113.9,36.1 L114.3,36.1 L114.7,36.1 L115.0,36.1 L115.4,36.1 L115.8,36.1
 L116.2,36.1 L116.6,36.1 L116.9,36.1 L117.3,36.1 L117.7,36.1 L118.1,36.1 L118.5,36.1 L118.9,36.1
 L119.2,36.1 L119.6,36.1 L120.0,36.1 L120.4,36.1 L120.8,36.1 L121.2,36.1 L121.5,36.1 L121.9,36.1
 L122.3,36.1 L122.7,36.1 L123.1,36.1 L123.5,36.1 L123.8,36.1 L124.2,36.1 L124.6,36.1 L125.0,36.1
 L125.4,36.1 L125.8,36.1 L126.1,36.1 L126.5,36.1 L126.9,36.2 L127.3,36.2 L127.7,36.2 L128.1,36.2
 L128.4,36.2 L128.8,36.2 L129.2,36.2 L129.6,36.2 L130.0,36.2 L130.3,36.2 L130.7,36.2 L131.1,36.2
 L131.5,36.2 L131.9,36.2 L132.3,36.2 L132.6,36.2 L133.0,36.2 L133.4,36.2 L133.8,36.2 L134.2,36.2
 L134.6,36.2 L134.9,36.2 L135.3,36.2 L135.7,36.2 L136.1,36.2 L136.5,36.2 L136.9,36.2 L137.2,36.2
 L137.6,36.2 L138.0,36.2 L138.4,36.2 L138.8,36.3 L139.2,36.3 L139.5,36.3 L139.9,36.3 L140.3,36.3
 L140.7,36.3 L141.1,36.3 L141.4,36.3 L141.8,36.3 L142.2,36.3 L142.6,36.3 L143.0,36.3 L143.4,36.3
 L143.7,36.3 L144.1,36.3 L144.5,36.4 L144.9,36.4 L145.3,36.4 L145.7,36.4 L146.0,36.4 L146.4,36.4
 L146.8,36.4 L147.2,36.4 L147.6,36.4 L148.0,36.4 L148.3,36.5 L148.7,36.5 L149.1,36.5 L149.5,36.5
 L149.9,36.5 L150.3,36.5 L150.6,36.5 L151.0,36.6 L151.4,36.6 L151.8,36.6 L152.2,36.6 L152.5,36.6
 L152.9,36.6 L153.3,36.7 L153.7,36.7 L154.1,36.7 L154.5,36.7 L154.8,36.7 L155.2,36.8 L155.6,36.8
 L156.0,36.8 L156.4,36.8 L156.8,36.8 L157.1,36.9 L157.5,36.9 L157.9,36.9 L158.3,36.9 L158.7,37.0
 L159.1,37.0 L159.4,37.0 L159.8,37.1 L160.2,37.1 L160.6,37.1 L161.0,37.2 L161.4,37.2 L161.7,37.2
 L162.1,37.3 L162.5,37.3 L162.9,37.3 L163.3,37.4 L163.7,37.4 L164.0,37.4 L164.4,37.5 L164.8,37.5
 L165.2,37.6 L165.6,37.6 L165.9,37.7 L166.3,37.7 L166.7,37.8 L167.1,37.8 L167.5,37.9 L167.9,37.9
 L168.2,38.0 L168.6,38.0 L169.0,38.1 L169.4,38.1 L169.8,38.2 L170.2,38.2 L170.5,38.3 L170.9,38.4
 L171.3,38.4 L171.7,38.5 L172.1,38.6 L172.5,38.6 L172.8,38.7 L173.2,38.8 L173.6,38.9 L174.0,38.9
 L174.4,39.0 L174.8,39.1 L175.1,39.2 L175.5,39.3 L175.9,39.3 L176.3,39.4 L176.7,39.5 L177.0,39.6
 L177.4,39.7 L177.8,39.8 L178.2,39.9 L178.6,40.0 L179.0,40.1 L179.3,40.2 L179.7,40.3 L180.1,40.4
 L180.5,40.5 L

In [24]:
GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(benchmarkData, 2, dataRef).ToGnuplot().PlotSVG()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 1.1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 center of mass - y 
 
 
 
 
 time 
 
 
 
 
 paper 
 
 
 
 
 paper 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M544.1,57.1 L597.5,57.1 M71.9,470.1 L72.3,470.1 L72.7,470.1 L73.1,470.1 L73.5,470.1 L73.8,470.1
 L74.2,470.0 L74.6,470.0 L75.0,470.0 L75.4,470.0 L75.8,470.0 L76.2,470.0 L76.6,470.0 L76.9,470.0
 L77.3,469.9 L77.7,469.9 L78.1,469.9 L78.5,469.9 L78.9,469.8 L79.3,469.8 L79.7,469.8 L80.0,469.8
 L80.4,469.7 L80.8,469.7 L81.2,469.7 L81.6,469.6 L82.0,469.6 L82.4,469.5 L82.8,469.5 L83.1,469.5
 L83.5,469.4 L83.9,469.4 L84.3,469.3 L84.7,469.3 L85.1,469.2 L85.5,469.2 L85.9,469.1 L86.2,469.1
 L86.6,469.0 L87.0,469.0 L87.4,468.9 L87.8,468.9 L88.2,468.8 L88.6,468.7 L89.0,468.7 L89.3,468.6
 L89.7,468.5 L90.1,468.5 L90.5,468.4 L90.9,468.3 L91.3,468.3 L91.7,468.2 L92.1,468.1 L92.4,468.0
 L92.8,468.0 L93.2,467.9 L93.6,467.8 L94.0,467.7 L94.4,467.7 L94.8,467.6 L95.2,467.5 L95.5,467.4
 L95.9,467.3 L96.3,467.2 L96.7,467.1 L97.1,467.0 L97.5,467.0 L97.9,466.9 L98.3,466.8 L98.6,466.7
 L99.0,466.6 L99.4,466.5 L99.8,466.4 L100.2,466.3 L100.6,466.2 L101.0,466.1 L101.4,466.0 L101.7,465.9
 L102.1,465.7 L102.5,465.6 L102.9,465.5 L103.3,465.4 L103.7,465.3 L104.1,465.2 L104.5,465.1 L104.8,465.0
 L105.2,464.8 L105.6,464.7 L106.0,464.6 L106.4,464.5 L106.8,464.4 L107.2,464.2 L107.6,464.1 L107.9,464.0
 L108.3,463.9 L108.7,463.7 L109.1,463.6 L109.5,463.5 L109.9,463.3 L110.3,463.2 L110.7,463.1 L111.0,462.9
 L111.4,462.8 L111.8,462.6 L112.2,462.5 L112.6,462.4 L113.0,462.2 L113.4,462.1 L113.8,461.9 L114.1,461.8
 L114.5,461.6 L114.9,461.5 L115.3,461.3 L115.7,461.2 L116.1,461.0 L116.5,460.9 L116.9,460.7 L117.2,460.6
 L117.6,460.4 L118.0,460.3 L118.4,460.1 L118.8,459.9 L119.2,459.8 L119.6,459.6 L120.0,459.5 L120.3,459.3
 L120.7,459.1 L121.1,459.0 L121.5,458.8 L121.9,458.6 L122.3,458.5 L122.7,458.3 L123.1,458.1 L123.4,457.9
 L123.8,457.8 L124.2,457.6 L124.6,457.4 L125.0,457.2 L125.4,457.1 L125.8,456.9 L126.2,456.7 L126.5,456.5
 L126.9,456.3 L127.3,456.1 L127.7,456.0 L128.1,455.8 L128.5,455.6 L128.9,455.4 L129.3,455.2 L129.6,455.0
 L130.0,454.8 L130.4,454.6 L130.8,454.4 L131.2,454.2 L131.6,454.0 L132.0,453.8 L132.4,453.6 L132.7,453.4
 L133.1,453.2 L133.5,453.0 L133.9,452.8 L134.3,452.6 L134.7,452.4 L135.1,452.2 L135.5,452.0 L135.8,451.8
 L136.2,451.6 L136.6,451.4 L137.0,451.2 L137.4,451.0 L137.8,450.7 L138.2,450.5 L138.6,450.3 L138.9,450.1
 L139.3,449.9 L139.7,449.7 L140.1,449.4 L140.5,449.2 L140.9,449.0 L141.3,448.8 L141.7,448.6 L142.0,448.3
 L142.4,448.1 L142.8,447.9 L143.2,447.7 L143.6,447.4 L144.0,447.2 L144.4,447.0 L144.8,446.7 L145.1,446.5
 L145.5,446.3 L145.9,446.0 L146.3,445.8 L146.7,445.6 L147.1,445.3 L147.5,445.1 L147.9,444.9 L148.2,444.6
 L148.6,444.4 L149.0,444.1 L149.4,443.9 L149.8,443.7 L150.2,443.4 L150.6,443.2 L151.0,442.9 L151.3,442.7
 L151.7,442.4 L152.1,442.2 L152.5,441.9 L152.9,441.7 L153.3,441.4 L153.7,441.2 L154.1,440.9 L154.4,440.7
 L154.8,440.4 L155.2,440.2 L155.6,439.9 L156.0,439.6 L156.4,439.4 L156.8,439.1 L157.2,438.9 L157.5,438.6
 L157.9,438.3 L158.3,438.1 L158.7,437.8 L159.1,437.6 L159.5,437.3 L159.9,437.0 L160.3,436.8 L160.6,436.5
 L161.0,436.2 L161.4,436.0 L161.8,435.7 L162.2,435.4 L162.6,435.2 L163.0,434.9 L163.4,434.6 L163.7,434.3
 L164.1,434.1 L164.5,433.8 L164.9,433.5 L165.3,433.2 L165.7,433.0 L166.1,432.7 L166.5,432.4 L166.8,432.1
 L167.2,431.8 L167.6,431.6 L168.0,431.3 L168.4,431.0 L168

In [25]:
GetBenchmarkQuantityComparisonOverTime_Plot2Ddata(benchmarkData, 3, dataRef, "rise velocity").ToGnuplot().PlotSVG()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.05 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rise velocity 
 
 
 
 
 time 
 
 
 
 
 paper 
 
 
 
 
 paper 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M544.1,57.1 L597.5,57.1 M80.2,458.0 L80.6,456.2 L81.0,454.4 L81.3,452.6 L81.7,450.8 L82.1,449.1
 L82.5,447.3 L82.9,445.5 L83.3,443.8 L83.6,442.0 L84.0,440.3 L84.4,438.5 L84.8,436.8 L85.2,435.0
 L85.6,433.3 L85.9,431.6 L86.3,429.9 L86.7,428.2 L87.1,426.4 L87.5,424.7 L87.9,423.0 L88.2,421.3
 L88.6,419.7 L89.0,418.0 L89.4,416.3 L89.8,414.6 L90.2,412.9 L90.5,411.3 L90.9,409.6 L91.3,407.9
 L91.7,406.3 L92.1,404.6 L92.4,403.0 L92.8,401.3 L93.2,399.7 L93.6,398.1 L94.0,396.4 L94.4,394.8
 L94.7,393.2 L95.1,391.5 L95.5,389.9 L95.9,388.3 L96.3,386.7 L96.7,385.1 L97.0,383.5 L97.4,381.9
 L97.8,380.3 L98.2,378.7 L98.6,377.1 L99.0,375.5 L99.3,373.9 L99.7,372.4 L100.1,370.8 L100.5,369.2
 L100.9,367.6 L101.3,366.1 L101.6,364.5 L102.0,363.0 L102.4,361.4 L102.8,359.9 L103.2,358.3 L103.6,356.8
 L103.9,355.2 L104.3,353.7 L104.7,352.2 L105.1,350.6 L105.5,349.1 L105.8,347.6 L106.2,346.0 L106.6,344.5
 L107.0,343.0 L107.4,341.5 L107.8,340.0 L108.1,338.5 L108.5,337.0 L108.9,335.5 L109.3,334.0 L109.7,332.5
 L110.1,331.0 L110.4,329.5 L110.8,328.0 L111.2,326.6 L111.6,325.1 L112.0,323.6 L112.4,322.2 L112.7,320.7
 L113.1,319.2 L113.5,317.8 L113.9,316.3 L114.3,314.9 L114.7,313.4 L115.0,312.0 L115.4,310.5 L115.8,309.1
 L116.2,307.6 L116.6,306.2 L116.9,304.8 L117.3,303.3 L117.7,301.9 L118.1,300.5 L118.5,299.1 L118.9,297.7
 L119.2,296.2 L119.6,294.8 L120.0,293.4 L120.4,292.0 L120.8,290.6 L121.2,289.2 L121.5,287.8 L121.9,286.5
 L122.3,285.1 L122.7,283.7 L123.1,282.3 L123.5,280.9 L123.8,279.6 L124.2,278.2 L124.6,276.8 L125.0,275.5
 L125.4,274.1 L125.8,272.7 L126.1,271.4 L126.5,270.0 L126.9,268.7 L127.3,267.3 L127.7,266.0 L128.1,264.7
 L128.4,263.3 L128.8,262.0 L129.2,260.7 L129.6,259.3 L130.0,258.0 L130.3,256.7 L130.7,255.4 L131.1,254.1
 L131.5,252.8 L131.9,251.5 L132.3,250.2 L132.6,248.9 L133.0,247.6 L133.4,246.3 L133.8,245.0 L134.2,243.7
 L134.6,242.4 L134.9,241.1 L135.3,239.9 L135.7,238.6 L136.1,237.3 L136.5,236.1 L136.9,234.8 L137.2,233.6
 L137.6,232.3 L138.0,231.1 L138.4,229.8 L138.8,228.6 L139.2,227.3 L139.5,226.1 L139.9,224.9 L140.3,223.6
 L140.7,222.4 L141.1,221.2 L141.4,220.0 L141.8,218.8 L142.2,217.6 L142.6,216.3 L143.0,215.1 L143.4,213.9
 L143.7,212.8 L144.1,211.6 L144.5,210.4 L144.9,209.2 L145.3,208.0 L145.7,206.8 L146.0,205.7 L146.4,204.5
 L146.8,203.3 L147.2,202.2 L147.6,201.0 L148.0,199.9 L148.3,198.7 L148.7,197.6 L149.1,196.4 L149.5,195.3
 L149.9,194.2 L150.3,193.0 L150.6,191.9 L151.0,190.8 L151.4,189.7 L151.8,188.6 L152.2,187.5 L152.5,186.3
 L152.9,185.2 L153.3,184.2 L153.7,183.1 L154.1,182.0 L154.5,180.9 L154.8,179.8 L155.2,178.7 L155.6,177.7
 L156.0,176.6 L156.4,175.5 L156.8,174.5 L157.1,173.4 L157.5,172.4 L157.9,171.3 L158.3,170.3 L158.7,169.2
 L159.1,168.2 L159.4,167.2 L159.8,166.2 L160.2,165.1 L160.6,164.1 L161.0,163.1 L161.4,162.1 L161.7,161.1
 L162.1,160.1 L162.5,159.1 L162.9,158.1 L163.3,157.1 L163.7,156.2 L164.0,155.2 L164.4,154.2 L164.8,153.2
 L165.2,152.3 L165.6,151.3 L165.9,150.4 L166.3,149.4 L166.7,148.5 L167.1,147.5 L167.5,146.6 L167.9,145.7
 L168.2,144.8 L168.6,143.8 L169.0,142.9 L169.4,142.0 L169.8,141.1 L170.2,140.2 L170.5,139.3 L170.9,138.4
 L171.3,137.5 L171.7,136.6 L172.1,135.8 L172.5,134.9 L172.8,134.0 L173.2,133.2 L173.6,132.3 L174.0,131.4
 L174.4,130.6 L174.8,129.7 L175.1,128.9 L175.5,128.1 L175.9,

## Error norms

In [26]:
using MathNet.Numerics.Interpolation;

In [27]:
// l1-error norm
public static (double l1_errNorm, double l2_errNorm, double lInf_errNorm) ComputeErrorNorms(Plot2Ddata.XYvalues data, Plot2Ddata.XYvalues refData) {

    double q_i = 0.0;
    double qRef_i = 0.0;
    double[] absc = data.Abscissas;
    double[] dqS = new double[absc.Length];
    double[]qRefS = new double[absc.Length];
    LinearSpline LinSplineRef = LinearSpline.InterpolateSorted(refData.Abscissas, refData.Values);
    for (int i = 0; i < absc.Length; i++) {
        q_i = data.Values[i];
        qRef_i = LinSplineRef.Interpolate(absc[i]);
        dqS[i] = Math.Abs(q_i - qRef_i);
        qRefS[i] = Math.Abs(qRef_i);
    }

    double l1err = dqS.Sum() / qRefS.Sum();
    double l2err = (dqS.L2NormPow2() / qRefS.L2NormPow2()).Sqrt();  
    double lInferr = dqS.Max() / qRefS.Max();

    return (l1err, l2err, lInferr);
}

In [28]:
// l1-error norm
public static (double[] time, double[] errors) GetErrorOverTime(Plot2Ddata.XYvalues data, Plot2Ddata.XYvalues refData) {

    double q_i = 0.0;
    double qRef_i = 0.0;
    double[] absc = data.Abscissas;
    double[] dqS = new double[absc.Length];
    LinearSpline LinSplineRef = LinearSpline.InterpolateSorted(refData.Abscissas, refData.Values);
    for (int i = 0; i < absc.Length; i++) {
        q_i = data.Values[i];
        qRef_i = LinSplineRef.Interpolate(absc[i]);
        dqS[i] = Math.Abs(q_i - qRef_i);
    }

    return (absc, dqS);
}

In [29]:
int qId = 2;
var bmDat = benchmarkData.ElementAt(qId).dataGroups[0];
var refDat = dataRef[qId, 0].dataGroups[0];

In [30]:
var res = GetErrorOverTime(bmDat, refDat);

In [31]:
Plot2Ddata plt =  new Plot2Ddata();
plt.Xlabel = "time";
plt.Ylabel = "error";

var fmt = new PlotFormat();
fmt.Style = Styles.Lines; 
fmt.DashType = DashTypes.Solid;
fmt.LineWidth = 2;
fmt.LineColor = LineColors.Red;

plt.AddDataGroup(res.time, res.errors, fmt);

In [32]:
plt.ToGnuplot().PlotSVG()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 error 
 
 
 
 
 time 
 
 
 
 
 gnuplot_plot_1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M88.5,542.4 L88.9,542.3 L89.4,542.2 L89.8,542.2 L90.3,542.1 L90.7,542.1 L91.1,542.0 L91.6,542.0
 L92.0,541.9 L92.5,541.8 L92.9,541.7 L93.4,541.7 L93.8,541.5 L94.2,541.5 L94.7,541.4 L95.1,541.3
 L95.6,541.2 L96.0,541.1 L96.4,541.0 L96.9,540.9 L97.3,540.8 L97.8,540.7 L98.2,540.6 L98.6,540.5
 L99.1,540.5 L99.5,540.4 L100.0,540.3 L100.4,540.2 L100.8,540.2 L101.3,540.1 L101.7,540.1 L102.2,540.0
 L102.6,540.0 L103.1,539.9 L103.5,539.9 L103.9,539.9 L104.4,539.9 L104.8,539.9 L105.3,539.9 L105.7,539.9
 L106.1,540.0 L106.6,540.1 L107.0,540.0 L107.5,540.1 L107.9,540.1 L108.3,540.1 L108.8,540.1 L109.2,540.0
 L109.7,540.0 L110.1,539.9 L110.6,539.8 L111.0,539.6 L111.4,539.5 L111.9,539.3 L112.3,539.1 L112.8,539.0
 L113.2,538.8 L113.6,538.6 L114.1,538.4 L114.5,538.3 L115.0,538.2 L115.4,538.1 L115.8,538.0 L116.3,538.0
 L116.7,538.0 L117.2,538.0 L117.6,538.1 L118.1,538.2 L118.5,538.3 L118.9,538.4 L119.4,538.5 L119.8,538.6
 L120.3,538.7 L120.7,538.8 L121.1,538.7 L121.6,538.6 L122.0,538.4 L122.5,538.2 L122.9,538.0 L123.3,537.8
 L123.8,537.6 L124.2,537.4 L124.7,537.2 L125.1,537.1 L125.5,537.1 L126.0,537.1 L126.4,537.2 L126.9,537.3
 L127.3,537.4 L127.8,537.6 L128.2,537.7 L128.6,537.8 L129.1,537.8 L129.5,537.8 L130.0,537.7 L130.4,537.5
 L130.8,537.3 L131.3,537.0 L131.7,536.8 L132.2,536.6 L132.6,536.5 L133.0,536.4 L133.5,536.4 L133.9,536.5
 L134.4,536.6 L134.8,536.7 L135.3,536.9 L135.7,537.1 L136.1,537.1 L136.6,537.1 L137.0,537.0 L137.5,536.8
 L137.9,536.6 L138.3,536.4 L138.8,536.2 L139.2,536.1 L139.7,536.0 L140.1,536.1 L140.5,536.1 L141.0,536.2
 L141.4,536.3 L141.9,536.4 L142.3,536.4 L142.8,536.4 L143.2,536.3 L143.6,536.2 L144.1,536.1 L144.5,536.0
 L145.0,535.9 L145.4,535.8 L145.8,535.8 L146.3,535.8 L146.7,535.7 L147.2,535.8 L147.6,535.8 L148.0,535.7
 L148.5,535.6 L148.9,535.6 L149.4,535.6 L149.8,535.7 L150.2,535.7 L150.7,535.7 L151.1,535.7 L151.6,535.6
 L152.0,535.4 L152.5,535.2 L152.9,535.1 L153.3,535.0 L153.8,535.0 L154.2,535.0 L154.7,535.1 L155.1,535.4
 L155.5,535.5 L156.0,535.6 L156.4,535.5 L156.9,535.3 L157.3,534.9 L157.7,534.5 L158.2,534.3 L158.6,534.3
 L159.1,534.4 L159.5,534.8 L160.0,535.2 L160.4,535.5 L160.8,535.4 L161.3,535.2 L161.7,534.8 L162.2,534.3
 L162.6,533.9 L163.0,533.8 L163.5,533.9 L163.9,534.2 L164.4,534.7 L164.8,535.0 L165.2,535.1 L165.7,534.8
 L166.1,534.5 L166.6,534.0 L167.0,533.7 L167.5,533.7 L167.9,533.7 L168.3,534.0 L168.8,534.2 L169.2,534.3
 L169.7,534.3 L170.1,534.1 L170.5,533.9 L171.0,533.8 L171.4,533.8 L171.9,533.7 L172.3,533.7 L172.7,533.6
 L173.2,533.4 L173.6,533.3 L174.1,533.3 L174.5,533.5 L174.9,533.8 L175.4,533.9 L175.8,533.8 L176.3,533.5
 L176.7,533.0 L177.2,532.6 L177.6,532.4 L178.0,532.7 L178.5,533.2 L178.9,533.6 L179.4,533.7 L179.8,533.5
 L180.2,533.0 L180.7,532.5 L181.1,532.1 L181.6,532.2 L182.0,532.6 L182.4,532.9 L182.9,533.0 L183.3,532.9
 L183.8,532.7 L184.2,532.5 L184.7,532.4 L185.1,532.4 L185.5,532.4 L186.0,532.2 L186.4,531.9 L186.9,531.8
 L187.3,532.0 L187.7,532.3 L188.2,532.6 L188.6,532.6 L189.1,532.2 L189.5,531.6 L189.9,531.1 L190.4,531.1
 L190.8,531.5 L191.3,531.9 L191.7,532.2 L192.2,532.0 L192.6,531.6 L193.0,531.2 L193.5,531.1 L193.9,531.1
 L194.4,531.2 L194.8,531.0 L195.2,530.8 L195.7,530.8 L196.1,531.0 L196.6,531.3 L197.0,531.3 L197.4,531.0
 L197.9,530.4 L198.3,529.9 L198.8,529.9 L199.2,530.3 L199.6,530.

In [33]:
public static void CheckMaxAllowedErrorNorms(List<Plot2Ddata> data, Plot2Ddata[,] refData, 
                                                int datIdx, double maxL1, double maxL2, double maxLInf) {
    var dataElement = data.ElementAt(datIdx);
    Console.WriteLine("{0}:", dataElement.Ylabel);
    for (int i = 0; i < dataElement.dataGroups.Count(); i++) {
        Console.WriteLine("Checking - {0}:", dataElement.dataGroups[i].Name);
        double maxl1_errNorm = 0.0; double maxl2_errNorm = 0.0; double maxlInf_errNorm = 0.0;

        for (int g = 0; g < refData.GetLength(1); g++) {
            var ret = ComputeErrorNorms(dataElement.dataGroups[i], refData[datIdx, g].dataGroups[0]);
            maxl1_errNorm = (ret.l1_errNorm > maxl1_errNorm) ? ret.l1_errNorm : maxl1_errNorm;
            maxl2_errNorm = (ret.l2_errNorm > maxl2_errNorm) ? ret.l2_errNorm : maxl2_errNorm;
            maxlInf_errNorm = (ret.lInf_errNorm > maxlInf_errNorm) ? ret.lInf_errNorm : maxlInf_errNorm;
        }

        Console.WriteLine("max l1-error norm: {0}, max l2-error norm: {1}, max lInf-error norm: {2}", 
        maxl1_errNorm, maxl2_errNorm, maxlInf_errNorm);

        NUnit.Framework.Assert.LessOrEqual(maxl1_errNorm, maxL1, 
                                    $"L1-error Norm for {dataElement.Ylabel} greater than allowed.");
        NUnit.Framework.Assert.LessOrEqual(maxl2_errNorm, maxL2, 
                                    $"L2-error Norm for {dataElement.Ylabel} greater than allowed.");
        NUnit.Framework.Assert.LessOrEqual(maxlInf_errNorm, maxLInf, 
                                    $"Linf-error Norm for {dataElement.Ylabel} greater than allowed.");
    }
}

In [34]:
CheckMaxAllowedErrorNorms(benchmarkData, dataRef, 1, 0.0025, 0.003, 0.005);

circularity:
Checking - RB2D-fullDomain-20x40AMR0-k3-Picard-FastMarching-ReInit50-testcase1:
max l1-error norm: 0.0022607383766853686, max l2-error norm: 0.002857342796546946, max lInf-error norm: 0.004642202376675453
Checking - RB2D-fullDomain-20x40AMR0-k3-StokesExt-ReInit300-semiImplicit-dt0.001-testcase1:
max l1-error norm: 0.0023541461610419225, max l2-error norm: 0.002897939344358424, max lInf-error norm: 0.004346615655851371


In [35]:
CheckMaxAllowedErrorNorms(benchmarkData, dataRef, 2, 0.0035, 0.004, 0.0055);

center of mass - y:
Checking - RB2D-fullDomain-20x40AMR0-k3-Picard-FastMarching-ReInit50-testcase1:
max l1-error norm: 0.003123699606812848, max l2-error norm: 0.003856667338161839, max lInf-error norm: 0.005132185078469803
Checking - RB2D-fullDomain-20x40AMR0-k3-StokesExt-ReInit300-semiImplicit-dt0.001-testcase1:
max l1-error norm: 0.004273472210350324, max l2-error norm: 0.00518022395378402, max lInf-error norm: 0.007014310910854852


Error: NUnit.Framework.AssertionException:   L1-error Norm for center of mass - y greater than allowed.
  Expected: less than or equal to 0.0035000000000000001d
  But was:  0.0042734722103503243d

   at NUnit.Framework.Assert.ReportFailure(String message)
   at NUnit.Framework.Assert.ReportFailure(ConstraintResult result, String message, Object[] args)
   at NUnit.Framework.Assert.That[TActual](TActual actual, IResolveConstraint expression, String message, Object[] args)
   at NUnit.Framework.Assert.LessOrEqual(Double arg1, Double arg2, String message, Object[] args)
   at Submission#33.CheckMaxAllowedErrorNorms(List`1 data, Plot2Ddata[,] refData, Int32 datIdx, Double maxL1, Double maxL2, Double maxLInf)
   at Submission#35.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHold

In [36]:
CheckMaxAllowedErrorNorms(benchmarkData, dataRef, 3, 0.011, 0.012, 0.015);

mean velocity - y:
Checking - RB2D-fullDomain-20x40AMR0-k3-Picard-FastMarching-ReInit50-testcase1:
max l1-error norm: 0.010873142052173067, max l2-error norm: 0.011801979696124568, max lInf-error norm: 0.0145761083138005
Checking - RB2D-fullDomain-20x40AMR0-k3-StokesExt-ReInit300-semiImplicit-dt0.001-testcase1:
max l1-error norm: 0.008550667741349806, max l2-error norm: 0.00924013701969605, max lInf-error norm: 0.011567841639119336


## Check values at specifc points in time

### minimum circularity and instance of time

In [37]:
public static (double circMin, double time) GetMinimumCircularity(Plot2Ddata.XYvalues circData) {
    double circMin = circData.Values[0];
    int minIndex = 0;
    for (int i = 1; i < circData.Values.Length; ++i) {
        if (circData.Values[i] < circMin) {
            circMin = circData.Values[i];
            minIndex = i;
        }
    }

    return (circMin, circData.Abscissas[minIndex]);
}

In [38]:
public static (double mean_MinCirc, double mean_MinCircTime) GetRefMeanMinCirc(Plot2Ddata[,] refData) {
    double mean_MinCirc = 0.0;
    double mean_MinCircTime = 0.0;
    for (int g = 0; g < refData.GetLength(1); g++) {
        var ret = GetMinimumCircularity(refData[1, g].dataGroups[0]);
        mean_MinCirc += ret.circMin;
        mean_MinCircTime += ret.time;
    }
    mean_MinCirc /= refData.GetLength(1);
    mean_MinCircTime /= refData.GetLength(1);

    return (mean_MinCirc, mean_MinCircTime);
}

In [ ]:
public static void CheckRelativeErrorFromRefMeanMinCirc(List<Plot2Ddata> data, Plot2Ddata[,] refData, double maxRelDev_MinCirc, double maxRelDev_MinCircTime) {

    var retRef = GetRefMeanMinCirc(refData);
    Console.WriteLine("Reference mean minimum circularity: {0} at time {1}", retRef.mean_MinCirc, retRef.mean_MinCircTime);

    // set max relative deviation from reference
    double maxRefDev_MinCirc = 0.0;
    double maxRefDev_MinCircTime = 0.0;
    for (int g = 0; g < refData.GetLength(1); g++) {
        var ret = GetMinimumCircularity(refData[1, g].dataGroups[0]);
        maxRefDev_MinCirc = Math.Max(maxRefDev_MinCirc, Math.Abs(ret.circMin - retRef.mean_MinCirc));
        maxRefDev_MinCircTime = Math.Max(maxRefDev_MinCircTime, Math.Abs(ret.time - retRef.mean_MinCircTime));
    }
    maxRefDev_MinCirc /= retRef.mean_MinCirc;
    maxRefDev_MinCircTime /= retRef.mean_MinCircTime;
    Console.WriteLine("Max relative deviation from reference mean minimum circularity: {0}, time: {1}", 
                        maxRefDev_MinCirc, maxRefDev_MinCircTime);

    var dataElement = data.ElementAt(1);
    for (int i = 0; i < dataElement.dataGroups.Count(); i++) {
        Console.WriteLine("Checking - {0}:", dataElement.dataGroups[i].Name);

        var ret = GetMinimumCircularity(data.ElementAt(1).dataGroups[i]);
        Console.WriteLine("Minimum circularity: {0} at time {1}", ret.circMin, ret.time);

        NUnit.Framework.Assert.LessOrEqual(Math.Abs(ret.circMin - retRef.mean_MinCirc)/retRef.mean_MinCirc, maxRelDev_MinCirc, $"Relative deviation from minimal circularity value greater than allowed.");
        NUnit.Framework.Assert.LessOrEqual(Math.Abs(ret.time - retRef.mean_MinCircTime)/retRef.mean_MinCircTime, maxRelDev_MinCircTime, $"Relative deviation from minimal circularity time greater than allowed.");
    }
    
}

In [46]:
CheckRelativeErrorFromRefMeanMinCirc(benchmarkData, dataRef, 0.004, 0.01);

Reference mean minimum circularity: 0.9011981099999998 at time 1.8930059999999997
Max relative deviation from reference mean minimum circularity: 0.00011338239490963951, time: 0.009511855746891318
Checking - RB2D-fullDomain-20x40AMR0-k3-Picard-FastMarching-ReInit50-testcase1:
Mean minimum circularity: 0.9047610196768266 at time 1.8760000000000014
Checking - RB2D-fullDomain-20x40AMR0-k3-StokesExt-ReInit300-semiImplicit-dt0.001-testcase1:
Mean minimum circularity: 0.9049421749207351 at time 1.8729999999999045


Error: NUnit.Framework.AssertionException:   Relative deviation from minimal circularity value greater than allowed.
  Expected: less than or equal to 0.0040000000000000001d
  But was:  0.0041545414700606877d

   at NUnit.Framework.Assert.ReportFailure(String message)
   at NUnit.Framework.Assert.ReportFailure(ConstraintResult result, String message, Object[] args)
   at NUnit.Framework.Assert.That[TActual](TActual actual, IResolveConstraint expression, String message, Object[] args)
   at NUnit.Framework.Assert.LessOrEqual(Double arg1, Double arg2, String message, Object[] args)
   at Submission#45.CheckRelativeErrorFromRefMeanMinCirc(List`1 data, Plot2Ddata[,] refData, Double maxRelDev_MinCirc, Double maxRelDev_MinCircTime)
   at Submission#46.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, Str

### Terminal rise height

In [47]:
public static double GetTerminalRiseHeight(Plot2Ddata.XYvalues yData) {
    return yData.Values.Last();
}

In [49]:
public static double GetRefMeanTerminalHeight(Plot2Ddata[,] refData) {

    double mean_tHeight = 0.0;
    for (int g = 0; g < refData.GetLength(1); g++) {
        mean_tHeight += GetTerminalRiseHeight(refData[2, g].dataGroups[0]);
    }
    mean_tHeight /= refData.GetLength(1);

    return mean_tHeight;
}

In [50]:
public static void CheckRelativeErrorFromRefMeanRiseHeight(List<Plot2Ddata> data, Plot2Ddata[,] refData, double maxRelDev_tHeight) {

    double mean_tHeight = GetRefMeanTerminalHeight(refData);
    Console.WriteLine("Reference mean terminal rise height: {0}", mean_tHeight);

    // set max relative deviation from reference
    double maxRefDev_tHeight = 0.0;
    for (int g = 0; g < refData.GetLength(1); g++) {
        var ret = GetTerminalRiseHeight(refData[2, g].dataGroups[0]);
        maxRefDev_tHeight = Math.Max(maxRefDev_tHeight, Math.Abs(ret - mean_tHeight));
    }
    maxRefDev_tHeight /= mean_tHeight;
    Console.WriteLine("Max relative deviation from reference mean terminal rise height: {0}", maxRefDev_tHeight);

    var dataElement = data.ElementAt(2);
    for (int i = 0; i < dataElement.dataGroups.Count(); i++) {
        Console.WriteLine("Checking - {0}:", dataElement.dataGroups[i].Name);

        var ret = GetTerminalRiseHeight(data.ElementAt(2).dataGroups[i]);
        Console.WriteLine("Terminal rise height: {0}", ret);

        NUnit.Framework.Assert.LessOrEqual(Math.Abs(ret - mean_tHeight)/mean_tHeight, maxRelDev_tHeight, $"Relative deviation from terminal rise height greater than allowed.");
    }
    
}

In [51]:
CheckRelativeErrorFromRefMeanRiseHeight(benchmarkData, dataRef, 0.005);

Reference mean terminal rise height: 1.08104536
Max relative deviation from reference mean terminal rise height: 0.0010853198611390047
Checking - RB2D-fullDomain-20x40AMR0-k3-Picard-FastMarching-ReInit50-testcase1:
Terminal rise height: 1.0762011709261912
Checking - RB2D-fullDomain-20x40AMR0-k3-StokesExt-ReInit300-semiImplicit-dt0.001-testcase1:
Terminal rise height: 1.0741651757977086


Error: NUnit.Framework.AssertionException:   Relative deviation from terminal rise height greater than allowed.
  Expected: less than or equal to 0.0050000000000000001d
  But was:  0.0063643806789859553d

   at NUnit.Framework.Assert.ReportFailure(String message)
   at NUnit.Framework.Assert.ReportFailure(ConstraintResult result, String message, Object[] args)
   at NUnit.Framework.Assert.That[TActual](TActual actual, IResolveConstraint expression, String message, Object[] args)
   at NUnit.Framework.Assert.LessOrEqual(Double arg1, Double arg2, String message, Object[] args)
   at Submission#50.CheckRelativeErrorFromRefMeanRiseHeight(List`1 data, Plot2Ddata[,] refData, Double maxRelDev_tHeight)
   at Submission#51.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Fun

### Maximum rise velocity and instance of time

In [52]:
public static (double riseVelMax, double time) GetMaximumRiseVelocity(Plot2Ddata.XYvalues riseData) {
    double riseMax = riseData.Values[0];
    int maxIndex = 0;
    for (int i = 1; i < riseData.Values.Length; ++i) {
        if (riseData.Values[i] > riseMax) {
            riseMax = riseData.Values[i];
            maxIndex = i;
        }
    }

    return (riseMax, riseData.Abscissas[maxIndex]);
}

In [53]:
public static (double mean_MaxRiseV, double mean_MaxRiseVtime) GetRefMeanMaxRiseV(Plot2Ddata[,] refData) {
    double mean_MaxRiseV = 0.0;
    double mean_MaxRiseVtime = 0.0;
    for (int g = 0; g < refData.GetLength(1); g++) {
        var ret = GetMaximumRiseVelocity(refData[3, g].dataGroups[0]);
        mean_MaxRiseV += ret.riseVelMax;
        mean_MaxRiseVtime += ret.time;
    }
    mean_MaxRiseV /= refData.GetLength(1);
    mean_MaxRiseVtime /= refData.GetLength(1);

    return (mean_MaxRiseV, mean_MaxRiseVtime);
}

In [54]:
public static void CheckMaxRiseVFromRefMeanMaxRiseV(List<Plot2Ddata> data, Plot2Ddata[,] refData, double maxRelDev_MaxRiseV, double maxRelDev_MaxRiseVTime) {

    var retRef = GetRefMeanMaxRiseV(refData);
    Console.WriteLine("Reference mean maximum rise velocity: {0} at time {1}", retRef.mean_MaxRiseV, retRef.mean_MaxRiseVtime);

    // set max relative deviation from reference
    double maxRefDev_MaxRiseV = 0.0;
    double maxRefDev_MaxRiseVTime = 0.0;
    for (int g = 0; g < refData.GetLength(1); g++) {
        var ret = GetMaximumRiseVelocity(refData[3, g].dataGroups[0]);
        maxRefDev_MaxRiseV = Math.Max(maxRefDev_MaxRiseV, Math.Abs(ret.riseVelMax - retRef.mean_MaxRiseV));
        maxRefDev_MaxRiseVTime = Math.Max(maxRefDev_MaxRiseVTime, Math.Abs(ret.time - retRef.mean_MaxRiseVtime));
    }
    maxRefDev_MaxRiseV /= retRef.mean_MaxRiseV;
    maxRefDev_MaxRiseVTime /= retRef.mean_MaxRiseVtime;
    Console.WriteLine("Max relative deviation from reference mean maximum rise velocity: {0}, time: {1}", 
                        maxRefDev_MaxRiseV, maxRefDev_MaxRiseVTime);

    var dataElement = data.ElementAt(3);
    for (int i = 0; i < dataElement.dataGroups.Count(); i++) {
        Console.WriteLine("Checking - {0}:", dataElement.dataGroups[i].Name);

        var ret = GetMaximumRiseVelocity(data.ElementAt(3).dataGroups[i]);
        Console.WriteLine("Maximum rise velocity: {0} at time {1}", ret.riseVelMax, ret.time);

        NUnit.Framework.Assert.LessOrEqual(Math.Abs(ret.riseVelMax - retRef.mean_MaxRiseV)/retRef.mean_MaxRiseV, maxRelDev_MaxRiseV, $"Relative deviation from maximum rise velocity greater than allowed.");
        NUnit.Framework.Assert.LessOrEqual(Math.Abs(ret.time - retRef.mean_MaxRiseVtime)/retRef.mean_MaxRiseVtime, maxRelDev_MaxRiseVTime, $"Relative deviation from maximum rise velocity time greater than allowed.");
    }
}

In [56]:
CheckMaxRiseVFromRefMeanMaxRiseV(benchmarkData, dataRef, 0.008, 0.015);

Reference mean maximum rise velocity: 0.24181627000000003 at time 0.9254658333333334
Max relative deviation from reference mean maximum rise velocity: 0.001153520397945008, time: 0.006250005627796369
Checking - RB2D-fullDomain-20x40AMR0-k3-Picard-FastMarching-ReInit50-testcase1:
Maximum rise velocity: 0.23995703954071837 at time 0.9120000000000007
Checking - RB2D-fullDomain-20x40AMR0-k3-StokesExt-ReInit300-semiImplicit-dt0.001-testcase1:
Maximum rise velocity: 0.2404931932923751 at time 0.9140000000000007


## Check terminal interface shape

In [57]:
public static (double[] x, double[] y) GetTerminalShapeInterfacePoints(ISessionInfo sess) {

    var terminalStep = sess.Timesteps.Last();
    DGField phi = terminalStep.GetField("Phi");
    LevelSet LevSet = new LevelSet(phi.Basis, "LevelSet"); 
    LevSet.Acc(1.0, phi);           
    LevelSetTracker LsTrk = new LevelSetTracker((BoSSS.Foundation.Grid.Classic.GridData) phi.GridDat, CutCellQuadratureMethod.Saye, 1, new string[] { "A", "B" }, LevSet);
    LsTrk.UpdateTracker(terminalStep.PhysicalTime);

    MultidimensionalArray interP = BoSSS.Solution.XNSECommon.XNSEUtils.GetInterfacePoints(LsTrk, LevSet, quadRuleOrderForNodeSet:10);
    double[] x = interP.ExtractSubArrayShallow(new int[] { -1, 0 }).To1DArray();
    double[] y = interP.ExtractSubArrayShallow(new int[] { -1, 1 }).To1DArray();

    return (x, y);
}

In [61]:
Plot2Ddata datShape = new Plot2Ddata();

foreach (var evalS in evalSess) {
    var shape = GetTerminalShapeInterfacePoints(evalS);

    var fmt = new PlotFormat();
    fmt.Style = Styles.Lines;
    fmt.Style      = Styles.Points;
    fmt.PointType  = PointTypes.Dot;
    fmt.LineColor = (LineColors)(evalSess.IndexOf(evalS) + 1);

    datShape.AddDataGroup(LegendNames[evalSess.IndexOf(evalS)], shape.x, shape.y, fmt);
}

In [63]:
for (int grp = 1; grp <= groups.Length; grp++) {

    string dat  = datCase+grp+"l"+datLvl[grp - 1]+"s.txt";
    string path = (userName.Equals(@"FDY\JenkinsCI")) ? dat : @"Featflow_referenceData\" + dat;
    string[] lines = File.ReadAllLines(path);
    double[] x = new double[lines.Length];
    double[] y = new double[lines.Length];

    for (int i = 0; i < lines.Length; i++) {
        x[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[0]);            
        y[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[1]);      
    }   

    string datName = groups[grp-1]+"-l"+datLvl[grp - 1]+"s";

    var fmt = new PlotFormat();
    fmt.Style = Styles.Lines;
    fmt.Style      = Styles.Points;
    fmt.PointType  = PointTypes.Dot;
    fmt.LineColor = (LineColors)(9 - grp);

    // if (grp == 1)
    //     pltForm.LineColor = LineColors.Magenta;
    // if (grp == 2)
    //     pltForm.LineColor = LineColors.Green;
    // if (grp == 3)
    //     pltForm.LineColor = LineColors.Orange;

    datShape.AddDataGroup(datName, x, y , fmt);
}

In [64]:
datShape.ToGnuplot().PlotSVG()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 0.95 
 
 
 
 
 1 
 
 
 
 
 1.05 
 
 
 
 
 1.1 
 
 
 
 
 1.15 
 
 
 
 
 1.2 
 
 
 
 
 1.25 
 
 
 
 
 1.3 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 paper 
 
 
 paper 
 
 
 
 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

 

